# Выборочное исследование данных

## Упрощение чтения

Сами по себе json'ы очень сложные для простого человеческого чтения.
Структуры очень вложенные, но хотелось бы понять, какие top level атрибуты есть у каждой сущности.

Сформируем набор файлов с постфиксом `_keys`, для того, чтобы иметь представление о том, какие атрибуты есть у каждой сущности.

In [ ]:
# imports
import os
import json
import json_utils
from pathlib import Path

In [ ]:
def prepare_for_serialization(value: dict) -> dict:
    serialized_data = {}

    for key, value in value.items():
        if isinstance(value, set):
            class_obj = next(iter(value))
            type_name = class_obj.__name__  # extract just 'int', 'str', etc.
            serialized_data[key] = type_name
    
    return serialized_data

def generate_compact_entity_preview(input_path: str, artifacts_path: str):
    os.makedirs(artifacts_path, exist_ok=True)
    # Get all files in folder
    for filename in os.listdir(input_path):
        file_path = os.path.join(input_path, filename)
        
        # Check if it's a file (not a subfolder)
        if os.path.isfile(file_path):
            entity_top_level_keys = json_utils.describe_entity_from_json(file_path)
            entity_keys_serialized = prepare_for_serialization(entity_top_level_keys)

            # Save to new file
            output_filename = f"{os.path.splitext(filename)[0]}_keys.json"
            output_path = os.path.join(artifacts_path, output_filename)
            
            with open(output_path, 'w') as f:
                f.write(json.dumps(entity_keys_serialized, indent=2))
            
            print(f"Processed: {filename} -> {output_filename}")

In [ ]:
# define path constants
INPUT_DATA_PATH = '../data'
ARTIFACTS_DATA_PATH = '../artifacts/structures'

In [ ]:
generate_compact_entity_preview(input_path=INPUT_DATA_PATH, artifacts_path=ARTIFACTS_DATA_PATH)

## Постановка задачи

Сущностей очень много.
Нужно попытаться понять, можем ли мы на таком наборе данных соединять разные сущности между собой.

Анализируя вручную, было принято решение посмотреть на связку между `persons` и `organisational-units`.

### Организации

#### Подготовка данных

Начнем работать с ними, т.к. у организации меньше полей, с которыми надо разобраться.

Согласно [документации Pure](https://helpcenter.pure.elsevier.com/organisational-unit):

> Organisational units are a research institution’s schools, faculties, institutes, departments, and so on; any type of unit that makes up an organisation.
> If these units are organised hierarchically, you can model the structure in Pure.

In [ ]:
# load items from json
organizations = json_utils.load_from_json(os.path.join(INPUT_DATA_PATH, 'organisational-units.json'))
organizations

Смотря на результаты `organisational-units_keys.json` и на данные в `organisational-units.json`, можно проигнорировать некоторые поля на данный момент.

Интересуют следующие поля - `uuid`, `level`, `parents`.

`uuid` - уникальный идентификатор сущности.
По всей видимости, можно вытаскивать связи между сущностями при помощи `uuid`, а не по `pureId`.

Про это написано в [документации Pure](https://helpcenter.pure.elsevier.com/understanding-dependents-in-the-api):

> - UUIDs are always present in content and serve as unique identifiers.
> - The API does not use traditional database constraints like primary and foreign keys.
> - Instead, relationships are retrieved dynamically through the DEPENDENTS endpoints, which identify content that relies on a given record.
> This helps you understand dependencies between content types and how they are connected. 
> This allows you to create logical associations.

`level`, по всей видимости, указывает на место в иерархии.

`parents`, по всей видимости, содержит `uuid` родительских отделений.

In [ ]:
def extract_organization_data(organizations: list) -> list:
    # Return a list of organizational units (a more compact one)
    # basically we reduce the amount of fields we want to explore

    result = list()

    for organization in organizations:
        organization_unit = dict()

        organization_unit['uuid'] = organization['uuid'] # unit uuid

        # organization_unit['name'] = organization['name']['text'][0]['value'] # unit name (ru locale)
        # CAUTION: works if we are sure that there will always be ru_RU locale
        organization_names = organization['name']['text']
        for name in organization_names:
            if name['locale'] == 'ru_RU':
                organization_unit['name'] = name['value']
        
        organization_unit['level'] = organization['type']['term']['text'][0]['value'] # unit level

        # if we have parents, should add them:
        if 'parents' in organization.keys():
            parents = list()
            for parent in organization['parents']:
                parents.append(parent['uuid'])
            organization_unit['parents'] = parents
        
        result.append(organization_unit)
    
    return result

Т.к. мы сжимаем количество данных до формата, который удобно представить в виде читаемой в Jupyter Notebook структуры, подключим `pandas` для построения датафрейма.

Будем использовать это как in-memory альтернативу для построения запросов к нашей "базе данных".

In [ ]:
# imports
import pandas as pd

In [ ]:
organizations_reduced = extract_organization_data(organizations)
organization_df = pd.DataFrame(organizations_reduced)

In [ ]:
# use to see column types and memory usage
organization_df.info(verbose=True, show_counts=True)

In [ ]:
# take random sample of 7 elements too see the data
organization_df.sample(7)

Посмотрим, за что отвечает `level`.

Из каждой группы `level` вытащим по 3 записи, чтобы почитать их названия.

In [ ]:
# show 3 organizational units per level
organization_df.groupby('level').head(3)

`Level 0` есть только у одной сущности из присланной выборки.
И это сам СПБГУ.

Выборочно посмотрим на каждые из `level`, которые у нас есть в выборке.

In [ ]:
filtered_by_level = organization_df[organization_df['level'] == 'Level 1']
filtered_by_level

Что можно сказать про последующие уровни?

`level 1` - факультеты и аспирантура / ординатура (почему?), их `parent` - спбгу

`level 2` - кафедры и общие названия программ (?)

`level 3` - какие-то сектора и более детализированные образовательные программы

**Скорее всего тут можно выстроить иерархию, древовидную структуру с корнем в спбгу!**

#### Выбор более узкой задачи

Поскольку у нас есть ПМ-ПУ в выгрузке, можем задаться более конкретной задачей: получить все организации, которые связаны с пм-пу в данной выборке.

In [ ]:
apmath_uuid = '0435d70c-2eef-4944-90ed-649c9118ccac'
apmath_units = organization_df[
        organization_df['parents']
        .apply(
            lambda x: isinstance(x, list) and apmath_uuid in x
        )
    ]
apmath_units

In [ ]:
# save apmath units uuids to use them with persons
apmath_units_uuid = apmath_units['uuid']
apmath_units_uuid_list = apmath_units_uuid.tolist()

### Люди

#### Подготовка данных

Согласно [документации Pure](https://helpcenter.pure.elsevier.com/take-advantage-of-the-person-profile):

> Contains a researcher's name, name variants, titles, IDs, links and more.

In [ ]:
# load items from json
persons = json_utils.load_from_json(os.path.join(INPUT_DATA_PATH, 'persons.json'))
persons

Согласно документации Pure, имя человека можно выделить из объекта `name`.

Посмотрим на структуру объекта для первого человека из выборки

In [ ]:
# take name
persons[0]['name']

Хотелось бы узнать, как связаны между собой человек и организация.

Видимо, есть 2 основных типа организаций, к которым может принадлежать человек - `staffOrganisationAssociations` и `studentOrganisationAssociations`.
Предположительно, первый атрибут обозначает принадлежность человека к организации как сотрудника.

Это подтверждается [спецификацией Pure](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml#/person/person_get):

> Organizations that the person is associated with as 'Staff'

Исходя из результатов в `persons_keys.json`, таких организаций у одного пользователя может быть несколько.
Посмотрим, как это можно вызвать из кода:

In [ ]:
# can take uuid of organisation
persons[0]['staffOrganisationAssociations'][0]['organisationalUnit']

Получается, `person` содержит в себе довольно подробную информацию об организации, а не только ее `uuid`.

Опять же, все поля сейчас просматривать не имеет смысла.
Ограничимся `uuid`, `name` и `uuid` организаций, с которыми связан пользователь.

In [ ]:
def extract_person_data(persons: list) -> list:
    # i want to take person uuid, name, organizational unit
    result = list()

    for person in persons:
        person_unit = dict()

        person_unit['uuid'] = person['uuid'] # unit uuid
        person_unit['name'] = f"{person['name']['lastName']} {person['name']['firstName']}"
        
        person_organizations = person['staffOrganisationAssociations']
        organization_uuids = list()
        organization_jobs = list()

        for organization in person_organizations:
            organization_uuids.append(organization['organisationalUnit']['uuid'])

            # CAUTION: doesn't work since jobTitle is not a required field
            # job_title_names = organization['jobTitle']['term']['text']
            # print(job_title_names)
            # for job in job_title_names:
            #     if job['locale'] == 'ru_RU':
            #         organization_jobs.append(job['value'])

        person_unit['organizations'] = organization_uuids
        result.append(person_unit)
    
    return result

Сформируем pandas dataframe для дальнейшей работы

In [ ]:
persons_reduced = extract_person_data(persons)
person_df = pd.DataFrame(persons_reduced)

In [ ]:
# get random sample of persons
person_df.sample(7)

#### Выбор более узкой задачи

Ранее мы уже нашли `uuid`s, которые относятся к подразделениям пм-пу.
Теперь посмотрим, есть ли в нашей выборке сотрудники из этих подразделений.

In [ ]:
apmath_persons = person_df[
        person_df['organizations']
        .apply(
            lambda x: isinstance(x, list) and any(uuid in x for uuid in apmath_units_uuid_list)
        )
    ]
apmath_persons

## Выводы

### Ценность текущих результатов

Как видно, на полученной выборке можно получать агрегированные данные.

Структура сущностей получается довольно громоздкой, потому что при наличии связей с другими сущностями включается не только связка в виде `uuid`, но и более подробная информация, которая затрудняет быстрый разбор и интерпретацию атрибутов, особенно при отсутствии документации.

В спецификации, представленной в вики репозитория, отсутствует описание ответов с сервера, поэтому изначально пришлось выстраивать предположения о том, что значит тот или иной атрибут, исходя из информации на Pure Helpdesk.

Лишь в конце исследования удалось найти общую спецификацию к Pure, которая довольно подробно описывает ответы в Pure API - [ссылка](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml).

**Поскольку это общая документация к версии API, которая выше версии в вики (5.22 vs 5.35), то к ней надо относиться с настороженностью.**
**Однако, это может сильно помочь для дальнейшего понимания.**

Из-за отсутствия спецификации на конкретную имплементацию сервиса, неясно, какие атрибуты будут присутствовать всегда (т.е. они `required`), а какие - нет (`optional`).
По этой причине не удалось вытащить должность сотрудника для подразделения - у кого-то такая информация есть, а у кого-то ее нет!

### Проблема ручного парсинга

Ручной парсинг json'ов - неприятное занятие.
Если хочется и дальше получать ответы с сервера, то нужно точно определить, какие поля нам нужно оставлять, а от каких отказываться.

Исходя из этого понимания, можно будет использовать методы, предоставляемые различными библиотеками.
Возможно, это позволит создать более fault tolerant решение.

### Что делать дальше?

Пока ручное построение запросов в pandas работает, оно не очень удобно для дальнейшей работы - держать несколько pandas dataframes одновременно в ram может быть проблематичным.
Возможно, стоит перейти к связке "бд + сервис запроса"

В целом, реляционная бд может подойти, но требуется понять, как организовать схему БД для укладывания существующих сущностей.
Исходя из увиденнного, один человек может принадлежать нескольким организационным подразделениям.
А организационные подразделения выстраивают иерархию.

Надо более подробно изучить ВСЕ сущности и понять связи между ними (напр. найти или сформировать ERM-диаграмму), определить, от каких атрибутов можно отказаться.

Наверное, это позволить сделать сущности более плоскими и удобными для укладки в БД и дальнейшего исследования данных.

## Источники для дальнейшего изучения

### Интернет-ресурсы

Список ресурсов, которые можно использовать для дальнейшей работы:

- [Pure Help Center: Documentation](https://helpcenter.pure.elsevier.com/en_US/documentation)
- [Pure API User Guide](https://helpcenter.pure.elsevier.com/pure-api-home)
- [Understanding Dependents in the Pure API](https://helpcenter.pure.elsevier.com/understanding-dependents-in-the-api)
- [Differences Between UUID and Pure ID](https://helpcenter.pure.elsevier.com/difference-between-uuid-and-pure-id)
- [Swagger: Pure API Specification](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml)
- [Overview of Content Types](https://helpcenter.pure.elsevier.com/overview-of-content-types)
- [Organisational Unit](https://helpcenter.pure.elsevier.com/organisational-unit)

### Литература и общение с AI по теме задачи

#### Pure от Elseveir - это CRIS / RIMS

Некоторые заметки по тому, что такое Pure в принципе.
Это будет полезно для поиска статей, посвященных таким системам.

Согласно [Википедии](https://en.wikipedia.org/wiki/Current_research_information_system):

> CRIS — это база данных или иная информационная система для хранения, управления и обмена контекстными метаданными об исследовательской деятельности, финансируемой исследовательским фондом или проводимой в организации, выполняющей исследования (или их объединении).

CRIS — это синоним RIMS (Research Information Management System, Система управления исследовательской информацией).
Pure — одна из таких систем.

### BI для RIMS

Википедия также дает краткую информацию о стороне Business Intelligence для RIMS:

> Благодаря комплексной агрегации контекстной исследовательской информации CRIS являются очень подходящими инструментами для извлечения показателей бизнес-аналитики для принятия решений в учреждениях и за их пределами.

Таким образом, мы можем попытаться найти статьи о принятии решений с использованием RIMS/CRIS.
Кроме того, я полагаю, что [SciVal](https://www.scival.com/landing) от Elsevier существует именно для этой цели.

Нашла немного литературы по этой теме:

- Статья Никифоровой с элементами предиктивного анализа (машинное обучение и статистика)
- Нашла обзорный PDF по Scival; предлагает несколько use cases для применения метрик

SciVal похож на то, что хотелось бы получить на выходе.
Вот что Gemini говорит о Scival:

> SciVal — это веб-аналитическое решение от Elsevier, использующее данные Scopus для визуализации, анализа и бенчмаркинга исследовательской эффективности более 20 000 учреждений, 10 000+ исследовательских тем и связанных с ними исследователей из более чем 230 стран.
>
> Оно обеспечивает стратегическое принятие решений, выявление партнеров и отслеживание тенденций, например, в области ЦУР (SDGs) и научных направлениях.
>
> Ключевые особенности и возможности:
>
> - **Бенчмаркинг (Сравнительный анализ):** Сравнивайте исследовательскую эффективность (публикационная активность, цитируемость, влияние) учреждений, команд или отдельных лиц с конкурентами, используя такие показатели, как FWCI (Field-Weighted Citation Impact — взвешенный по области науки показатель цитирования).
> - **Сотрудничество:** Определяйте существующих или потенциальных партнеров путем анализа сетей соавторства и поиска ведущих экспертов в конкретных областях.
> - **Анализ трендов:** Изучайте исследовательские тренды, тематические кластеры и актуальные темы для выявления новых областей исследований.
> - **Отчетность:** Создавайте настраиваемые отчеты для демонстрации научного влияния (research impact) в целях получения финансирования, найма сотрудников или участия в рейтингах.
> - **Источники данных:** Использует данные Scopus, охватывающие период с 1996 года по настоящее время и включающие более 80 миллионов записей от 7000+ издателей.

Итак... еще одна тема для ресерча: что такое бенчмаркинг и какие метрики существуют в области CRIS и академической среды.